# Lab1：静态页面与 API 数据采集

姓名：沈昊辉　学号：112404260123　班级：数据1241

## 任务一：Device API 采集

`/devices` 页面数据为动态加载（源代码中搜不到设备名），API 为 `/api/devices`，参数 `page` / `page_size`。

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from collections import Counter
from pathlib import Path
import json

BASE = "https://115.159.159.82"
api_url = BASE + "/api/devices"

r = requests.get(api_url, params={"page": 1, "page_size": 50}, timeout=10)
print("status:", r.status_code)
print("url:", r.url)
print("content-type:", r.headers.get("Content-Type"))

In [ ]:
data = r.json()
print(type(data), data.keys())
print(data)

In [ ]:
# 一条真实设备记录，确认字段含义
print(data["items"][0])

## 任务二：分页采集全部设备

In [ ]:
total = data["total"]
page_size = data["page_size"]
total_pages = (total + page_size - 1) // page_size

print("total:", total)
print("page_size:", page_size)
print("total_pages:", total_pages)

In [ ]:
devices = []

for page in range(1, total_pages + 1):
    r = requests.get(
        api_url,
        params={"page": page, "page_size": page_size},
        timeout=10
    )
    page_data = r.json()
    devices.extend(page_data["items"])

print("device count:", len(devices))

## 任务三：Company 静态页面解析（第 1 页）

企业记录为 `<article class="company-card" data-company-id="...">`，静态加载。

In [ ]:
base_url = BASE
current_url = base_url + "/companies"

r = requests.get(current_url, timeout=10)
soup = BeautifulSoup(r.text, "html.parser")

cards = soup.select("article.company-card")
print("本页企业数：", len(cards))

In [ ]:
# 先解析第一条
card = cards[0]

company_id = card.get("data-company-id")
name = card.select_one("h2 a").get_text(strip=True)
status = card.select_one("span.badge").get_text(strip=True)

details = {}
for item in card.select("dl > div"):
    key = item.select_one("dt").get_text(strip=True)
    value = item.select_one("dd").get_text(strip=True)
    details[key] = value

industry = details["所属行业"]
zone = details["所在区域"]
device_count = int(details["设备数量"].replace("台", "").strip())

print(company_id, name, status, industry, zone, device_count)

In [ ]:
companies = []
for card in cards:
    details = {}
    for item in card.select("dl > div"):
        details[item.select_one("dt").get_text(strip=True)] = item.select_one("dd").get_text(strip=True)
    companies.append({
        "company_id": card.get("data-company-id"),
        "name": card.select_one("h2 a").get_text(strip=True),
        "industry": details["所属行业"],
        "zone": details["所在区域"],
        "device_count": int(details["设备数量"].replace("台", "").strip()),
        "status": card.select_one("span.badge").get_text(strip=True),
    })

print("本页企业数：", len(companies))

## 任务四：根据“下一页”连续采集 3 页

注意：第 2 页起“上一页”也是 `<a>` 链接，必须按文本“下一页”过滤，否则会翻回第 1 页。

In [ ]:
next_link = None
for a in soup.select("nav.pagination a"):
    if "下一页" in a.get_text(" ", strip=True):
        next_link = a
        break

print(next_link.get_text(" ", strip=True))
print(next_link.get("href"))

In [ ]:
companies = []
current_url = base_url + "/companies"

for i in range(3):
    resp = requests.get(current_url, timeout=10)
    page_soup = BeautifulSoup(resp.text, "html.parser")
    cards = page_soup.select("article.company-card")

    for card in cards:
        details = {}
        for item in card.select("dl > div"):
            details[item.select_one("dt").get_text(strip=True)] = item.select_one("dd").get_text(strip=True)
        companies.append({
            "company_id": card.get("data-company-id"),
            "name": card.select_one("h2 a").get_text(strip=True),
            "industry": details["所属行业"],
            "zone": details["所在区域"],
            "device_count": int(details["设备数量"].replace("台", "").strip()),
            "status": card.select_one("span.badge").get_text(strip=True),
        })
    print(f"第 {i+1} 页：{len(cards)} 条")

    next_link = None
    for a in page_soup.select("nav.pagination a"):
        if "下一页" in a.get_text(" ", strip=True):
            next_link = a
            break
    if next_link is None or i == 2:
        break
    current_url = urljoin(current_url, next_link.get("href"))

print("Company 总数：", len(companies))

## 任务五：最终验证

（本单元格输出即报告【截图 4】）

In [ ]:
print("Company 总数:", len(companies))
print("唯一 company_id 数:", len(set(x["company_id"] for x in companies)))
print("Device 总数:", len(devices))
print("唯一 device_id 数:", len(set(x["device_id"] for x in devices)))

device_counts = Counter(x["company_id"] for x in devices)
mismatch_count = 0
for company in companies:
    if company["device_count"] != device_counts.get(company["company_id"], 0):
        mismatch_count += 1
print("Company-Device 不一致企业数:", mismatch_count)

## 任务六：保存采集结果

In [ ]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)

with open(data_dir / "companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("数据已保存到 data/ 目录")